# Phase 2 — Met-Seg Two-Stage CNN Baseline
## Fine-tuning BraTS-METS Pretrained Detector + Segmenter on Cyprus PROTEAS

**Paper:** Sadegheih & Merhof (MICCAI 2024 PRIME)

**Key Advantages over SegResNet:**
- Pretrained on **402 brain metastasis cases** (not gliomas!)
- Two-stage: DenseNet121 detector → DynUNet segmenter
- Designed for **small, scattered metastases**

**Repo:** [xmindflow/Met-Seg](https://github.com/xmindflow/Met-Seg)

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║  CHANGE THESE FOR EACH SESSION                           ║
# ╚════════════════════════════════════════════════════════════╝

MODE = 'train_all'   # 'quick_test' | 'train' | 'train_all'
FOLD = 0
CV_TYPE = '3fold'

# ╔════════════════════════════════════════════════════════════╗
# ║  HYPERPARAMETERS (matching Met-Seg paper)                ║
# ╚════════════════════════════════════════════════════════════╝

CONFIG = {
    'seg_params': {
        'spatial_dims': 3,
        'in_channels': 4,
        'out_channels': 3,
        'kernel_size': [[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3]],
        'strides': [[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
        'upsample_kernel_size': [[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
        'deep_supervision': True,
        'deep_supr_num': 3,
        'filters': [32, 64, 128, 256, 320],
        'res_block': True,
        'trans_bias': True,
    },
    'det_params': {
        'spatial_dims': 3,
        'in_channels': 4,
        'out_channels': 1,
        'dropout_prob': 0.2,
    },
    'patch_size': [64, 64, 64],
    'num_samples': 5,
    'pos_neg_ratio': [2, 1],
    'lr': 1e-4,              # Fine-tuning LR (pretrained weights)
    'weight_decay': 3e-5,
    'momentum': 0.99,
    'cache_rate': 0.5,
    'batch_size': 2,
    'num_workers': 0,
}

if MODE == 'quick_test':
    CONFIG['epochs'] = 5
    CONFIG['patience'] = 5
    CONFIG['val_interval'] = 1
else:
    CONFIG['epochs'] = 30           # 30 is enough with pretrained weights
    CONFIG['patience'] = 10
    CONFIG['val_interval'] = 5      # validate every 5 epochs to save time
    CONFIG['num_samples'] = 3       # 3 crops/volume instead of 5

REGION_NAMES = ['WT', 'TC', 'ET']

print(f'Mode: {MODE} | Fold: {FOLD} | CV: {CV_TYPE}')
print(f'Epochs: {CONFIG["epochs"]} | LR: {CONFIG["lr"]} | Patch: {CONFIG["patch_size"]}')
print(f'Segmenter: DynUNet | Detector: DenseNet121')
if MODE == 'train_all':
    print('Training ALL 3 folds + embeddings')

In [ ]:
# Install dependencies
import subprocess, sys
for pkg in ['monai', 'nibabel', 'pytorch-lightning', 'easydict']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies installed ✅')

In [ ]:
import os, shutil
from pathlib import Path

REPO_DIR = Path('/kaggle/working/Met-Seg')
if not REPO_DIR.exists():
    os.system('git clone https://github.com/xmindflow/Met-Seg.git /kaggle/working/Met-Seg')
    print('Cloned Met-Seg repo ✅')
else:
    print('Met-Seg repo already exists ✅')

import sys
sys.path.insert(0, str(REPO_DIR / 'src'))
print(f'Repo at: {REPO_DIR}')

In [ ]:
import os, subprocess
from pathlib import Path

OUTPUT_ROOT = Path('/kaggle/working/phase2_metseg_outputs')
WEIGHTS_DIR = OUTPUT_ROOT / 'pretrained_weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# Weight files we need
WEIGHT_FILES = {
    'segmentor': 'segmentor_full_modality.ckpt',
    'detector': 'detector_full_modality.ckpt',
}

WEIGHT_URLS = {
    'segmentor': 'https://myfiles.uni-regensburg.de/filr/public-link/file-download/0447879c90b809a80190bbf452df0f6b/120822/-2821463328806488135/segmentor_weight%20%28full%20modality%29.ckpt',
    'detector': 'https://myfiles.uni-regensburg.de/filr/public-link/file-download/0447879c90b809a80190bbf45b1d0f6f/120821/-4880761362689834300/detector_weight%20%28full%20modality%29.ckpt',
}

# ── Strategy 1: Check if weights exist in Kaggle dataset input ──
# (If you uploaded them as a separate dataset called 'metseg-pretrained-weights')
KAGGLE_WEIGHT_DIRS = [
    Path('/kaggle/input/datasets/zinou123viva/metseg-pretrained-weights'),
    Path('/kaggle/input/metseg-weights'),
    Path('/kaggle/input/met-seg-weights'),
]

def find_weight_in_kaggle(name):
    """Search for weight file in Kaggle dataset inputs."""
    for d in KAGGLE_WEIGHT_DIRS:
        if not d.exists():
            continue
        # Search recursively for .ckpt files
        for f in d.rglob('*.ckpt'):
            if name in f.name.lower() or name in f.stem.lower():
                return f
        # Also check direct children
        for f in d.iterdir():
            if f.suffix == '.ckpt':
                if name in f.name.lower():
                    return f
    return None

for name, filename in WEIGHT_FILES.items():
    dst = WEIGHTS_DIR / filename
    
    if dst.exists() and dst.stat().st_size > 1_000_000:
        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (cached)')
        continue
    
    # Strategy 1: Kaggle dataset input
    kaggle_path = find_weight_in_kaggle(name)
    if kaggle_path:
        import shutil
        shutil.copy2(kaggle_path, dst)
        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (from Kaggle dataset)')
        continue
    
    # Strategy 2: wget with browser User-Agent (uni-regensburg blocks urllib)
    url = WEIGHT_URLS[name]
    print(f'  Downloading {name} via wget...')
    result = subprocess.run(
        ['wget', '-q', '--no-check-certificate',
         '--user-agent', 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36',
         '-O', str(dst), url],
        capture_output=True, text=True, timeout=300
    )
    
    if result.returncode == 0 and dst.exists() and dst.stat().st_size > 1_000_000:
        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (downloaded)')
        continue
    
    # Strategy 3: curl as fallback
    print(f'  wget failed, trying curl...')
    result = subprocess.run(
        ['curl', '-L', '-s', '-o', str(dst),
         '-H', 'User-Agent: Mozilla/5.0 (X11; Linux x86_64)',
         url],
        capture_output=True, text=True, timeout=300
    )
    
    if result.returncode == 0 and dst.exists() and dst.stat().st_size > 1_000_000:
        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (downloaded via curl)')
        continue
    
    # Failed
    print(f'  ❌ Could not get {name} weights!')
    print(f'     Upload them as a Kaggle dataset called "metseg-pretrained-weights"')
    print(f'     containing: segmentor_full_modality.ckpt and detector_full_modality.ckpt')

print(f'\nWeights dir: {WEIGHTS_DIR}')
for f in WEIGHTS_DIR.glob('*.ckpt'):
    print(f'  {f.name}: {f.stat().st_size/1024/1024:.1f} MB')

In [ ]:
import warnings
# Suppress noisy MONAI and PyTorch warnings for clean logs
warnings.filterwarnings('ignore', message='.*Num foregrounds 0.*')
warnings.filterwarnings('ignore', message='.*non-tuple sequence for multidimensional indexing.*')
warnings.filterwarnings('ignore', message='.*axcodes.*length is smaller.*')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict
from tqdm import tqdm

from monai.networks.nets import DynUNet, DenseNet121
from monai.losses import DiceLoss
from torch.nn import BCEWithLogitsLoss
from monai.data import DataLoader, CacheDataset
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
import monai.transforms as T
from monai.transforms.compose import MapTransform
from monai.utils import ensure_tuple_rep

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# ── Data path resolution ──

DATA_ROOT = Path('/kaggle/input/datasets/zinou123viva/cyprus-proteas-brain-mets')
SYMLINK_DIR = Path('/kaggle/working/nifti_links')

def resolve_path(root, rel):
    p = root / rel
    if p.exists(): return str(p)
    gz = str(p) + '.gz'
    if Path(gz).exists(): return gz
    nii_gz = str(p).replace('.nii.gz', '.nii_gz')
    if Path(nii_gz).exists():
        link = SYMLINK_DIR / rel
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists(): os.symlink(nii_gz, str(link))
        return str(link)
    parent = p.parent
    if parent.exists():
        target = p.name
        for f in parent.iterdir():
            if f.name.lower() == target.lower(): return str(f)
            nii_gz_f = str(f).replace('.nii_gz', '.nii.gz')
            if Path(nii_gz_f).name.lower() == target.lower():
                link = SYMLINK_DIR / rel
                link.parent.mkdir(parents=True, exist_ok=True)
                if not link.exists(): os.symlink(str(f), str(link))
                return str(link)
    raise FileNotFoundError(f'Not found: {rel}')

# Try multiple possible splits file names
splits_file = None
for name in ['data_splits.json', 'cv_splits_3fold.json', 'cv_splits.json']:
    candidate = DATA_ROOT / name
    if candidate.exists():
        splits_file = candidate
        break
if splits_file is None:
    for f in DATA_ROOT.rglob('*splits*.json'):
        splits_file = f; break

with open(splits_file) as f:
    raw_splits = json.load(f)

# Handle nested structure: {'3fold': {'fold_0': ...}} vs flat {'fold_0': ...}
if '3fold' in raw_splits:
    all_splits = raw_splits['3fold']
elif 'fold_0' in raw_splits:
    all_splits = raw_splits
else:
    raise ValueError(f'Cannot find fold data. Keys: {list(raw_splits.keys())}')

print(f'Loaded splits: {splits_file.name}')
print(f'Folds: {list(all_splits.keys())}')

In [ ]:
def build_scan_dicts(scans, label='subset'):
    dicts, skips = [], 0
    for scan in scans:
        try:
            dicts.append({
                'image': [resolve_path(DATA_ROOT, scan['t1']), resolve_path(DATA_ROOT, scan['t1c']),
                          resolve_path(DATA_ROOT, scan['t2']), resolve_path(DATA_ROOT, scan['fla'])],
                'label': resolve_path(DATA_ROOT, scan['mask']),
                'patient_dir': scan['patient_dir'], 'visit': scan['visit'],
            })
        except FileNotFoundError: skips += 1
    if skips: print(f'  Skipped {skips} in {label}')
    return dicts

def get_fold_dicts(fold):
    fd = all_splits[f'fold_{fold}']
    return build_scan_dicts(fd['train_scans'], f'fold{fold}_train'), build_scan_dicts(fd['test_scans'], f'fold{fold}_val')

def get_all_dicts():
    all_d, seen = [], set()
    for fk in all_splits:
        for scan in all_splits[fk]['train_scans'] + all_splits[fk]['test_scans']:
            key = (scan['patient_dir'], scan['visit'])
            if key not in seen:
                seen.add(key)
                try:
                    all_d.append({'image': [resolve_path(DATA_ROOT, scan['t1']), resolve_path(DATA_ROOT, scan['t1c']),
                                            resolve_path(DATA_ROOT, scan['t2']), resolve_path(DATA_ROOT, scan['fla'])],
                                  'label': resolve_path(DATA_ROOT, scan['mask']),
                                  'patient_dir': scan['patient_dir'], 'visit': scan['visit']})
                except FileNotFoundError: pass
    return all_d

train_dicts, val_dicts = get_fold_dicts(0)
print(f'Fold 0: {len(train_dicts)} train | {len(val_dicts)} val')

In [ ]:
# ── Label conversion: Cyprus {0,1,2,3} → BraTS-METS [WT, TC, ET] ──
# Cyprus uses SAME labels as BraTS-METS! No remapping needed!

class ConvertToMultiChannelBratsMetsd(MapTransform):
    """Convert labels to 3-channel [WT, TC, ET] — Met-Seg convention."""
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            result = [
                (img == 1) | (img == 3) | (img == 2),  # WT
                (img == 1) | (img == 3),                # TC
                img == 3,                                # ET
            ]
            d[key] = torch.stack(result, dim=0).float() if isinstance(img, torch.Tensor) else np.stack(result, axis=0).astype(np.float32)
        return d

print('Label mapping (Cyprus → BraTS-METS):')
print('  Channel 0 (WT) = labels 1+2+3 (all tumor)')
print('  Channel 1 (TC) = labels 1+3 (tumor core)')
print('  Channel 2 (ET) = label 3 (enhancing only)')
print('  NO remapping needed ✅')

In [ ]:
# ── Transforms: exact Met-Seg augmentation pipeline ──

patch = CONFIG['patch_size']

train_transforms = T.Compose([
    T.LoadImaged(keys=['image', 'label']),
    T.EnsureChannelFirstd(keys=['image', 'label']),
    T.EnsureTyped(keys=['image', 'label']),
    T.Orientationd(keys=['image', 'label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image', 'label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBratsMetsd(keys=['label']),
    # Met-Seg augmentation
    T.RandAffined(keys=['image', 'label'],
        rotate_range=(np.deg2rad(30), np.deg2rad(30), np.deg2rad(30)),
        scale_range=((-0.3, 0.4), (-0.3, 0.4), (-0.3, 0.4)),
        padding_mode='zeros', mode=('trilinear', 'nearest'), prob=0.2),
    T.RandGaussianNoised(keys='image', prob=0.1, mean=0, std=0.1),
    T.RandGaussianSmoothd(keys='image', prob=0.2, sigma_x=(0.5, 1), sigma_y=(0.5, 1), sigma_z=(0.5, 1)),
    T.RandScaleIntensityd(keys='image', factors=0.25, prob=0.15),
    T.RandAdjustContrastd(keys='image', gamma=(0.7, 1.5), prob=0.3),
    T.RandFlipd(keys=['image', 'label'], spatial_axis=[0], prob=0.5),
    T.RandFlipd(keys=['image', 'label'], spatial_axis=[1], prob=0.5),
    T.RandFlipd(keys=['image', 'label'], spatial_axis=[2], prob=0.5),
    T.SpatialPadd(keys=['image', 'label'], spatial_size=ensure_tuple_rep(patch, 3)),
    T.RandCropByPosNegLabeld(keys=['image', 'label'], label_key='label',
        spatial_size=ensure_tuple_rep(patch, 3),
        pos=CONFIG['pos_neg_ratio'][0], neg=CONFIG['pos_neg_ratio'][1],
        num_samples=CONFIG['num_samples'], image_key='image', image_threshold=0),
    T.EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
])

val_transforms = T.Compose([
    T.LoadImaged(keys=['image', 'label']),
    T.EnsureChannelFirstd(keys=['image', 'label']),
    T.EnsureTyped(keys=['image', 'label']),
    T.Orientationd(keys=['image', 'label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image', 'label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBratsMetsd(keys=['label']),
    T.EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
])

print('Transforms defined ✅')
print(f'  Training: Met-Seg augmentation + RandCropByPosNegLabel (64³ patches)')
print(f'  {CONFIG["num_samples"]} random crops per volume')

In [ ]:
# ── Model creation & weight loading ──

def create_segmenter():
    return DynUNet(**CONFIG['seg_params'])

def create_detector():
    return DenseNet121(**CONFIG['det_params'])

def load_lightning_weights(model, ckpt_path, prefix='model.'):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd = ckpt.get('state_dict', ckpt)
    new_sd = OrderedDict()
    for k, v in sd.items():
        new_key = k[len(prefix):] if k.startswith(prefix) else k
        new_sd[new_key] = v
    missing, unexpected = model.load_state_dict(new_sd, strict=False)
    return missing, unexpected

# ── Find weight files ──
# Priority: WEIGHTS_DIR (from cell 4) → Kaggle input datasets
def find_weight(name):
    """Find a weight file by name."""
    # Check WEIGHTS_DIR first (from cell 4 download)
    candidate = WEIGHTS_DIR / f'{name}_full_modality.ckpt'
    if candidate.exists() and candidate.stat().st_size > 1_000_000:
        return candidate
    
    # Search Kaggle input directories
    import glob
    search_patterns = [
        f'/kaggle/input/metseg-pretrained-weights/**/{name}*.ckpt',
        f'/kaggle/input/metseg-pretrained-weights/**/*{name}*.ckpt',
        f'/kaggle/input/met-seg*/**/{name}*.ckpt',
        f'/kaggle/input/*metseg*/**/*{name}*.ckpt',
        f'/kaggle/input/**/{name}*modality*.ckpt',
    ]
    for pattern in search_patterns:
        matches = glob.glob(pattern, recursive=True)
        for m in matches:
            if os.path.getsize(m) > 1_000_000:
                return Path(m)
    
    return None

seg_weights = find_weight('segmentor')
det_weights = find_weight('detector')

print(f'Segmenter weights: {seg_weights}')
print(f'Detector weights:  {det_weights}')

# Test model creation + weight loading
seg_test = create_segmenter()
det_test = create_detector()

if seg_weights:
    print('\nLoading segmenter weights...')
    sm, su = load_lightning_weights(seg_test, seg_weights)
    print(f'  Missing: {len(sm)} | Unexpected: {len(su)}')
    if sm: print(f'  Missing keys: {sm[:3]}...')
else:
    print('\n⚠️ Segmenter weights not found — training from scratch')

if det_weights:
    print('Loading detector weights...')
    dm, du = load_lightning_weights(det_test, det_weights)
    print(f'  Missing: {len(dm)} | Unexpected: {len(du)}')
else:
    print('⚠️ Detector weights not found — training from scratch')

sp = sum(p.numel() for p in seg_test.parameters())
dp = sum(p.numel() for p in det_test.parameters())
print(f'\nSegmenter (DynUNet): {sp:,} params ({sp/1e6:.1f}M)')
print(f'Detector (DenseNet121): {dp:,} params ({dp/1e6:.1f}M)')
print(f'Total: {(sp+dp):,} params ({(sp+dp)/1e6:.1f}M)')

has_pretrained = seg_weights is not None and det_weights is not None
if has_pretrained:
    print('\n✅ BraTS-METS pretrained weights loaded successfully!')
else:
    print('\n⚠️ Some weights missing — check that you added "metseg-pretrained-weights" dataset')

del seg_test, det_test


In [ ]:
# ── Embedding hook for DynUNet bottleneck ──

_embedding_storage = {}

def _hook_fn(module, input, output):
    if isinstance(output, (list, tuple)): feat = output[0]
    else: feat = output
    _embedding_storage['feat'] = feat.detach()

def register_embedding_hook(model):
    # DynUNet: hook the last downsample block (320 channels)
    target = None
    if hasattr(model, 'downsamples'):
        target = model.downsamples[-1]
    if target is not None:
        hook = target.register_forward_hook(_hook_fn)
        print(f'  Hook registered on last downsample block ✅')
        return hook
    else:
        print('  ⚠️ Listing modules to find bottleneck:')
        for name, _ in model.named_children():
            print(f'    {name}')
        return None

print('Embedding hook ready')

In [ ]:
def train_fold(fold, resume_path=None):
    train_dicts, val_dicts = get_fold_dicts(fold)
    print(f'  Train: {len(train_dicts)} | Val: {len(val_dicts)}')
    
    train_ds = CacheDataset(train_dicts, train_transforms, cache_rate=CONFIG['cache_rate'], num_workers=CONFIG['num_workers'])
    val_ds = CacheDataset(val_dicts, val_transforms, cache_rate=1.0, num_workers=CONFIG['num_workers'])
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=CONFIG['num_workers'])
    
    seg_model = create_segmenter().to(device)
    det_model = create_detector().to(device)
    if seg_weights:
        load_lightning_weights(seg_model, seg_weights)
        print('  Loaded segmenter pretrained weights ✅')
    else:
        print('  ⚠️ No segmenter weights — training from scratch')
    if det_weights:
        load_lightning_weights(det_model, det_weights)
        print('  Loaded detector pretrained weights ✅')
    else:
        print('  ⚠️ No detector weights — detector from scratch')
    
    det_model.eval()
    for p in det_model.parameters(): p.requires_grad = False
    
    optimizer = torch.optim.SGD(seg_model.parameters(), lr=CONFIG['lr'],
        momentum=CONFIG['momentum'], weight_decay=CONFIG['weight_decay'], nesterov=True)
    dice_loss_fn = DiceLoss(sigmoid=True, smooth_nr=0, smooth_dr=1e-5)
    bce_loss_fn = BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler('cuda')
    dice_metric = DiceMetric(include_background=True, reduction='mean_batch')
    
    best_dice, patience_ctr = 0.0, 0
    metrics_log = {'train_loss': [], 'val_dice': [], 'val_per_region': []}
    t0 = time.time()
    
    for epoch in range(CONFIG['epochs']):
        seg_model.train()
        epoch_loss, n_steps = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            # Stage 1: Detector (frozen)
            with torch.no_grad():
                det_pred = det_model(images)
                tumor_mask = torch.sigmoid(det_pred.squeeze(-1)) > 0.3
            if tumor_mask.sum() == 0:
                tumor_mask = torch.ones(len(images), dtype=torch.bool, device=device)
            chosen_img = images[tumor_mask]
            chosen_lbl = labels[tumor_mask]
            
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = seg_model(chosen_img)
                if isinstance(outputs, (list, tuple)):
                    loss = sum(0.5**i * (dice_loss_fn(p, chosen_lbl) + bce_loss_fn(p, chosen_lbl)) for i, p in enumerate(outputs))
                elif outputs.dim() == 6:
                    preds = torch.unbind(outputs, dim=1)
                    loss = sum(0.5**i * (dice_loss_fn(p, chosen_lbl) + bce_loss_fn(p, chosen_lbl)) for i, p in enumerate(preds))
                else:
                    loss = dice_loss_fn(outputs, chosen_lbl) + bce_loss_fn(outputs, chosen_lbl)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(seg_model.parameters(), max_norm=12.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            n_steps += 1
        
        avg_loss = epoch_loss / max(n_steps, 1)
        metrics_log['train_loss'].append(avg_loss)
        
        # Validation
        if (epoch + 1) % CONFIG['val_interval'] == 0 or epoch == CONFIG['epochs'] - 1:
            seg_model.eval()
            dice_metric.reset()
            with torch.no_grad():
                for vb in val_loader:
                    vi = vb['image'].to(device)
                    vl = vb['label'].to(device)
                    vo = sliding_window_inference(vi, CONFIG['patch_size'], 4, seg_model, overlap=0.5, mode='gaussian')
                    if isinstance(vo, (list, tuple)): vo = vo[0]
                    if vo.dim() == 6: vo = vo[:, 0]
                    vp = (torch.sigmoid(vo) > 0.5).float()
                    dice_metric(vp, vl)
            dv = dice_metric.aggregate()
            md = dv.mean().item()
            pr = [dv[i].item() for i in range(len(REGION_NAMES))]
            metrics_log['val_dice'].append(md)
            metrics_log['val_per_region'].append(pr)
            elapsed = (time.time() - t0) / 60
            rs = ' '.join([f'{n}={v:.3f}' for n, v in zip(REGION_NAMES, pr)])
            print(f'Epoch {epoch:3d}/{CONFIG["epochs"]-1} | Loss: {avg_loss:.4f} | Dice: {md:.4f} ({rs}) | {elapsed:.1f} min')
            if md > best_dice:
                best_dice = md; patience_ctr = 0
                ckpt_dir = OUTPUT_ROOT / 'checkpoints'; ckpt_dir.mkdir(parents=True, exist_ok=True)
                torch.save({'model_state_dict': seg_model.state_dict(), 'epoch': epoch, 'best_dice': best_dice, 'config': CONFIG},
                           ckpt_dir / f'metseg_fold{fold}_best.pth')
                print(f'  ✅ New best! Saved.')
            else:
                patience_ctr += 1
                if patience_ctr >= CONFIG['patience']:
                    print(f'  Early stopping.'); break
        else:
            elapsed = (time.time() - t0) / 60
            print(f'Epoch {epoch:3d}/{CONFIG["epochs"]-1} | Loss: {avg_loss:.4f} | {elapsed:.1f} min')
    
    ckpt_dir = OUTPUT_ROOT / 'checkpoints'; ckpt_dir.mkdir(parents=True, exist_ok=True)
    torch.save({'model_state_dict': seg_model.state_dict(), 'epoch': epoch, 'best_dice': best_dice},
               ckpt_dir / f'metseg_fold{fold}_last.pth')
    elapsed = (time.time() - t0) / 60
    print(f'\n  Fold {fold} complete: Best Dice = {best_dice:.4f} | Time = {elapsed:.1f} min')
    
    fig_dir = OUTPUT_ROOT / 'figures'; fig_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(metrics_log['train_loss']); axes[0].set_title(f'Loss (Fold {fold})')
    if metrics_log['val_dice']: axes[1].plot(metrics_log['val_dice'], label='Mean Dice'); axes[1].legend()
    axes[1].set_title(f'Val Dice (Fold {fold})')
    plt.tight_layout(); plt.savefig(fig_dir / f'training_curves_fold{fold}.png', dpi=150); plt.close()
    
    met_dir = OUTPUT_ROOT / 'metrics'; met_dir.mkdir(parents=True, exist_ok=True)
    with open(met_dir / f'fold{fold}_metrics.json', 'w') as f: json.dump(metrics_log, f, indent=2)
    
    return seg_model, best_dice, metrics_log

In [ ]:
def extract_embeddings(model, fold):
    model.eval(); model.to(device)
    global _embedding_storage; _embedding_storage = {}
    emb_hook = register_embedding_hook(model)
    all_dicts = get_all_dicts()
    print(f'  Extracting from {len(all_dicts)} scans...')
    emb_ds = CacheDataset(all_dicts, val_transforms, cache_rate=CONFIG['cache_rate'], num_workers=CONFIG['num_workers'])
    emb_loader = DataLoader(emb_ds, batch_size=1, shuffle=False, num_workers=CONFIG['num_workers'])
    embeddings = {}
    with torch.no_grad():
        for i, batch in enumerate(tqdm(emb_loader, desc='Extracting')):
            images = batch['image'].to(device)
            patient = batch['patient_dir'][0]; visit = batch['visit'][0]
            key = f'{patient}__{visit}'
            _ = sliding_window_inference(images, CONFIG['patch_size'], 4, model, overlap=0.5, mode='gaussian')
            if 'feat' in _embedding_storage:
                feat = _embedding_storage['feat']
                emb = F.adaptive_avg_pool3d(feat, 1).flatten()
                embeddings[key] = emb.cpu().numpy()
    if emb_hook is not None: emb_hook.remove()
    if len(embeddings) == 0: print('  ⚠️ No embeddings!'); return
    emb_dir = OUTPUT_ROOT / 'embeddings'; emb_dir.mkdir(parents=True, exist_ok=True)
    emb_dim = list(embeddings.values())[0].shape[0]
    np.savez(emb_dir / f'cnn_metseg_embeddings_fold{fold}.npz', **embeddings)
    meta = {k: {'patient_dir': k.split('__')[0], 'visit': k.split('__')[1]} for k in embeddings}
    with open(emb_dir / f'cnn_metseg_embeddings_fold{fold}_meta.json', 'w') as f: json.dump(meta, f, indent=2)
    print(f'  ✅ {len(embeddings)} embeddings × {emb_dim}-dim saved')

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║  MAIN EXECUTION                                          ║
# ╚════════════════════════════════════════════════════════════╝

folds = [0, 1, 2] if MODE == 'train_all' else [FOLD]
results = {}

for fold in folds:
    print('=' * 60)
    print(f'  Training Fold {fold} | {MODE.upper()} | {CONFIG["epochs"]} epochs')
    print('=' * 60)
    
    model, best_dice, metrics = train_fold(fold)
    results[fold] = best_dice
    
    best_path = OUTPUT_ROOT / f'checkpoints/metseg_fold{fold}_best.pth'
    if best_path.exists():
        ckpt = torch.load(best_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
    
    print(f'\n  Extracting embeddings for fold {fold}...')
    extract_embeddings(model, fold)
    print()

print('=' * 60)
print('  SESSION COMPLETE')
print('=' * 60)
for fold, dice in results.items():
    print(f'  Fold {fold}: Dice={dice:.4f}')

print(f'\n  Files saved to {OUTPUT_ROOT}:')
for p in sorted(OUTPUT_ROOT.rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'    {p.relative_to(OUTPUT_ROOT)} ({size_kb:.1f} KB)')

print('\n  ⚠️ DOWNLOAD these files before session ends!')